# 04 CNN Training

Stage 7 prepares PyTorch-ready tensors for deep learning, and Stage 8 trains the first validation-monitored 1D CNN. This notebook starts with shape, leakage, and preprocessing checks for a single-epoch CNN dataset, then runs the tiny overfit smoke test and one-epoch training loop through reusable `src.train` utilities.

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import torch

from src.data import (
    DEFAULT_EPOCH_INDEX_PATH,
    DEFAULT_PREPROCESSING_METADATA_PATH,
    DEFAULT_RAW_DATA_DIR,
    DreamtContextDataset,
    DreamtEpochDataset,
    DreamtSequenceDataset,
    check_epoch_split_leakage,
    fit_normalization_stats,
    load_preprocessing_metadata,
    save_preprocessing_metadata,
)
from src.train import (
    DEFAULT_STAGE8_OUTPUT_DIR,
    TrainConfig,
    run_tiny_overfit_test,
    train_model,
)

CHANNELS = ["BVP", "ACC_X", "ACC_Y", "ACC_Z", "TEMP", "EDA", "HR", "IBI"]
BATCH_SIZE = 16
DEBUG_PARTICIPANTS = 3
EPOCHS = 1

raw_dir = repo_root / DEFAULT_RAW_DATA_DIR
epoch_index_path = repo_root / DEFAULT_EPOCH_INDEX_PATH
metadata_path = repo_root / DEFAULT_PREPROCESSING_METADATA_PATH
output_dir = repo_root / DEFAULT_STAGE8_OUTPUT_DIR
stage8_config = TrainConfig(
    raw_dir=raw_dir,
    epoch_index_path=epoch_index_path,
    preprocessing_metadata_path=metadata_path,
    output_dir=output_dir,
    channels=CHANNELS,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    max_train_participants=DEBUG_PARTICIPANTS,
    max_val_participants=DEBUG_PARTICIPANTS,
)

artifacts_available = raw_dir.exists() and epoch_index_path.exists()
artifacts_available


## Build Single-Epoch Datasets

In [ ]:
if artifacts_available:
    train_unscaled = DreamtEpochDataset(
        raw_dir=raw_dir,
        epoch_index=epoch_index_path,
        split="train",
        channels=CHANNELS,
        max_participants=DEBUG_PARTICIPANTS,
    )
    stats = fit_normalization_stats(train_unscaled)
    save_preprocessing_metadata(stats, metadata_path)
else:
    print("Skipping dataset construction because local raw files or epoch_index.csv are absent.")


In [ ]:
if artifacts_available:
    stats = load_preprocessing_metadata(metadata_path)
    train_ds = DreamtEpochDataset(raw_dir, epoch_index_path, split="train", channels=CHANNELS, preprocessing_stats=stats, max_participants=DEBUG_PARTICIPANTS)
    val_ds = DreamtEpochDataset(raw_dir, epoch_index_path, split="validation", channels=CHANNELS, preprocessing_stats=stats, max_participants=DEBUG_PARTICIPANTS)
    check_epoch_split_leakage(train_ds.epoch_index)
    check_epoch_split_leakage(val_ds.epoch_index)
    loaders = {
        "train": torch.utils.data.DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True),
        "validation": torch.utils.data.DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False),
    }
    x_batch, y_batch = next(iter(loaders["train"]))
    print("train batch:", tuple(x_batch.shape), x_batch.dtype, tuple(y_batch.shape), y_batch.dtype)
    print("participants:", {"train": len(train_ds.participants), "validation": len(val_ds.participants)})
    print("metadata channels:", stats["channels"])


## Temporal Context And Sequence Shape Checks

In [ ]:
if artifacts_available:
    context_ds = DreamtContextDataset(raw_dir, epoch_index_path, split="train", channels=CHANNELS, preprocessing_stats=stats, context_radius=2, max_participants=DEBUG_PARTICIPANTS)
    sequence_ds = DreamtSequenceDataset(raw_dir, epoch_index_path, split="train", channels=CHANNELS, preprocessing_stats=stats, sequence_length=5, label_mode="many_to_one", target_position="center", max_participants=DEBUG_PARTICIPANTS)
    if len(context_ds):
        x_context, y_context = context_ds[0]
        print("context item:", tuple(x_context.shape), y_context.item())
    if len(sequence_ds):
        x_sequence, y_sequence = sequence_ds[0]
        print("sequence item:", tuple(x_sequence.shape), y_sequence.item())


## Tiny Overfit Smoke Test

In [ ]:
if artifacts_available:
    overfit_history = run_tiny_overfit_test(train_ds, stage8_config)
    print("loss first/last:", round(overfit_history["loss"].iloc[0], 4), round(overfit_history["loss"].iloc[-1], 4))


## Single-Epoch CNN Training

In [ ]:
if artifacts_available:
    training_result = train_model(loaders["train"], loaders["validation"], stage8_config)
    display(training_result.history)
    print("best epoch:", training_result.best_epoch)
    print("outputs:", training_result.output_dir)
